### =============================================================================
## SECTION 1: IMPORTS, CONFIGURATION & DATA INGESTION
### =============================================================================

In [4]:
import numpy as np
import pandas as pd
import pymysql
from sqlalchemy import inspect, text
from core.database_utils import get_db_engine, get_database_inventory, run_query
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sqlalchemy import text

In [5]:

# Definir la consulta para listar tablas del esquema 'dbo' o todos
query_tables = text("""
    SELECT 
    s.name AS [Schema],
    t.name AS [Table],
    c.name AS [Column],
    pc.name AS [DataType],
    c.max_length AS [MaxLength],
    c.is_nullable AS [IsNullable]
FROM sys.columns c
JOIN sys.tables t ON c.object_id = t.object_id
JOIN sys.schemas s ON t.schema_id = s.schema_id
JOIN sys.types pc ON c.user_type_id = pc.user_type_id
WHERE t.name IN ('CATALOGO_CLIENTES_DATA', 'COBRANZA_DATA', 'CREDITO_DATA', 'DESTINOS_CLIENTES')
ORDER BY t.name, c.column_id;
""")

df_tab = run_query(query_tables, "Table inventory")
print(df_tab)


--- Success: [Table inventory] retrieved with 45 rows. ---
     Schema                   Table             Column  DataType  MaxLength  \
0   proadel  CATALOGO_CLIENTES_DATA             CODIGO  nvarchar         40   
1   proadel  CATALOGO_CLIENTES_DATA             NOMBRE  nvarchar        300   
2   proadel  CATALOGO_CLIENTES_DATA          DIRECCION  nvarchar        300   
3   proadel  CATALOGO_CLIENTES_DATA           TELEFONO  nvarchar        100   
4   proadel  CATALOGO_CLIENTES_DATA  LIMITE_DE_CREDITO   decimal          9   
5   proadel  CATALOGO_CLIENTES_DATA      TOTAL_CREDITO   decimal          9   
6   proadel  CATALOGO_CLIENTES_DATA     TOTAL_COBRANZA   decimal          9   
7   proadel  CATALOGO_CLIENTES_DATA    TOTAL_DEV_VENTA   decimal          9   
8   proadel  CATALOGO_CLIENTES_DATA      SALDO_INICIAL   decimal          9   
9   proadel  CATALOGO_CLIENTES_DATA             ESTADO  nvarchar        100   
10  proadel  CATALOGO_CLIENTES_DATA   TOTAL_POR_ABONAR   decimal        

In [6]:

# =============================================================================
# SECTION 1: IMPORTS, CONFIGURATION & DATA INGESTION
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from sqlalchemy import text

# Reference date for all recency/age calculations
ANALYSIS_DATE = pd.Timestamp.today().normalize()

# Global chart style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({
    "figure.dpi"     : 110,
    "axes.titlesize" : 13,
    "axes.labelsize" : 11,
    "legend.fontsize": 9,
})

# -- 1.1  Customer catalog ------------------------------------------------
df_customers = run_query(
    "SELECT * FROM proadel.CATALOGO_CLIENTES_DATA",
    "Customer Catalog"
)

# -- 1.2  Payment / collection records ------------------------------------
df_payments = run_query(
    "SELECT * FROM proadel.COBRANZA_DATA",
    "Payment (Cobranza) Data"
)

# -- 1.3  Credit / sales records ------------------------------------------
df_credits = run_query(
    "SELECT * FROM proadel.CREDITO_DATA",
    "Credit (Ventas) Data"
)

# -- 1.4  Customer-destination mapping ------------------------------------
df_destinations = run_query(
    "SELECT * FROM proadel.DESTINOS_CLIENTES",
    "Destinations Data"
)

print(f"\nShapes loaded:")
print(f"  Customers    : {df_customers.shape}")
print(f"  Payments     : {df_payments.shape}")
print(f"  Credits      : {df_credits.shape}")
print(f"  Destinations : {df_destinations.shape}")


--- Success: [Customer Catalog] retrieved with 404 rows. ---
--- Success: [Payment (Cobranza) Data] retrieved with 12022 rows. ---
--- Success: [Credit (Ventas) Data] retrieved with 16476 rows. ---
--- Success: [Destinations Data] retrieved with 35 rows. ---

Shapes loaded:
  Customers    : (404, 11)
  Payments     : (12022, 14)
  Credits      : (16476, 18)
  Destinations : (35, 2)


### =============================================================================
## SECTION 2: DATA CLEANING & TYPE NORMALIZATION
### =============================================================================

In [ ]:
# -- 2.1  Sanitize column names that contain spaces -----------------------
df_payments = df_payments.rename(columns={
    "FORMA DE PAGO"    : "FORMA_DE_PAGO",
    "CONCEPTO DIVERSO" : "CONCEPTO_DIVERSO",
})

# -- 2.2  Parse FECHA columns (stored as nvarchar in the source system) ---
df_payments["FECHA"] = pd.to_datetime(df_payments["FECHA"], dayfirst=True, errors="coerce")
df_credits["FECHA"]  = pd.to_datetime(df_credits["FECHA"],  dayfirst=True, errors="coerce")

# -- 2.3  Numeric coercion for catalog amount columns ---------------------
money_cols = [
    "LIMITE_DE_CREDITO", "TOTAL_CREDITO", "TOTAL_COBRANZA",
    "TOTAL_DEV_VENTA", "SALDO_INICIAL", "TOTAL_POR_ABONAR",
]
for col in money_cols:
    df_customers[col] = pd.to_numeric(df_customers[col], errors="coerce").fillna(0)

df_payments["IMPORTE"] = pd.to_numeric(df_payments["IMPORTE"], errors="coerce").fillna(0)
df_credits["TOTAL"]    = pd.to_numeric(df_credits["TOTAL"],    errors="coerce").fillna(0)
df_credits["IMPORTE"]  = pd.to_numeric(df_credits["IMPORTE"],  errors="coerce").fillna(0)



# -- 2.4  Drop cancelled records where applicable -------------------------
if "ESTADO" in df_payments.columns:
    df_payments = df_payments[
        df_payments["ESTADO"].str.strip().str.upper() != "CANCELADO"
    ].copy()
if "ESTADO" in df_credits.columns:
    df_credits = df_credits[
        df_credits["ESTADO"].str.strip().str.upper() != "CANCELADO"
    ].copy()

print(f"Clean records — Payments: {len(df_payments):,}  |  Credits: {len(df_credits):,}")
print(f"Date range — Payments : {df_payments['FECHA'].min().date()} → {df_payments['FECHA'].max().date()}")
print(f"Date range — Credits  : {df_credits['FECHA'].min().date()}  → {df_credits['FECHA'].max().date()}")


Clean records — Payments: 12,022  |  Credits: 16,476
Clean records — Payments: 12,022  |  Credits: 16,476
Date range — Payments : 2023-01-25 → 2026-03-27
Date range — Credits  : 2023-01-25  → 2026-03-28


### =============================================================================
## SECTION 3: CUSTOMER-LEVEL FEATURE ENGINEERING
### =============================================================================

3.1  Payment aggregates per customer (from COBRANZA_DATA)

In [8]:
pay_agg = (
    df_payments
    .groupby("CLIENTE", as_index=False)
    .agg(
        payment_count      = ("IMPORTE", "count"),
        total_payments     = ("IMPORTE", "sum"),
        avg_payment        = ("IMPORTE", "mean"),
        last_payment_date  = ("FECHA",   "max"),
        first_payment_date = ("FECHA",   "min"),
    )
)
pay_agg["days_since_last_payment"] = (
    ANALYSIS_DATE - pay_agg["last_payment_date"]
).dt.days.clip(lower=0)
pay_agg["payment_span_days"] = (
    (pay_agg["last_payment_date"] - pay_agg["first_payment_date"])
    .dt.days.clip(lower=1)
)
pay_agg["avg_days_between_payments"] = (
    pay_agg["payment_span_days"] / pay_agg["payment_count"].clip(lower=1)
)

print(pay_agg.head())

      CLIENTE  payment_count  total_payments    avg_payment last_payment_date  \
0   A.ESPARZA             44      5685934.74  129225.789545        2025-05-12   
1  A.ESPARZA3             23      1368180.92   59486.126957        2026-03-11   
2       AARON             69      1817085.91   26334.578406        2026-02-25   
3    AAVARIOS            206      7182098.36   34864.555146        2026-03-19   
4        ABEL             39     12960517.00  332320.948718        2025-02-26   

  first_payment_date  days_since_last_payment  payment_span_days  \
0         2023-02-03                      329                829   
1         2023-02-22                       26               1113   
2         2023-02-06                       40               1115   
3         2023-03-02                       18               1113   
4         2023-01-28                      404                760   

   avg_days_between_payments  
0                  18.840909  
1                  48.391304  
2          

3.2  Credit/sales aggregates per customer (from CREDITO_DATA)

In [9]:
cred_agg = (
    df_credits
    .groupby("CLIENTE", as_index=False)
    .agg(
        credit_count        = ("TOTAL",  "count"),
        total_purchases     = ("TOTAL",  "sum"),
        avg_invoice         = ("TOTAL",  "mean"),
        last_purchase_date  = ("FECHA",  "max"),
        first_purchase_date = ("FECHA",  "min"),
    )
)
cred_agg["recency_days"] = (
    ANALYSIS_DATE - cred_agg["last_purchase_date"]
).dt.days.clip(lower=0)
# Tenure in months between first and last purchase
cred_agg["tenure_months"] = (
    (cred_agg["last_purchase_date"] - cred_agg["first_purchase_date"])
    .dt.days / 30.44
).clip(lower=1)
cred_agg["monthly_purchase_rate"] = (
    cred_agg["total_purchases"] / cred_agg["tenure_months"]
)
print(cred_agg.head())

      CLIENTE  credit_count  total_purchases    avg_invoice  \
0   A.ESPARZA            49       5465304.92  111536.835102   
1  A.ESPARZA3            33       1353354.52   41010.743030   
2       AARON           180       1715579.44    9530.996889   
3    AAVARIOS           270       6330391.48   23445.894370   
4        ABEL            30       9060102.00  302003.400000   

  last_purchase_date first_purchase_date  recency_days  tenure_months  \
0         2025-05-28          2023-02-15           313      27.365309   
1         2026-03-10          2023-02-20            27      36.596583   
2         2026-02-28          2023-02-06            37      36.727989   
3         2026-03-19          2023-02-13            18      37.122208   
4         2025-09-01          2023-09-27           217      23.160315   

   monthly_purchase_rate  
0          199716.544736  
1           36980.351516  
2           46710.409797  
3          170528.421815  
4          391190.787064  


3.3  Merge into master feature table (customer catalog is the spine)

In [10]:
features = (
    df_customers[[
        "CODIGO", "NOMBRE", "ESTADO", "LIMITE_DE_CREDITO",
        "TOTAL_CREDITO", "TOTAL_COBRANZA", "SALDO_INICIAL", "TOTAL_POR_ABONAR"
    ]]
    .merge(pay_agg.rename(columns={"CLIENTE": "CODIGO"}),  on="CODIGO", how="left")
    .merge(cred_agg.rename(columns={"CLIENTE": "CODIGO"}), on="CODIGO", how="left")
)

# Fill zeros for customers with no transaction history
zero_cols = [
    "payment_count", "total_payments", "avg_payment",
    "credit_count", "total_purchases", "avg_invoice", "monthly_purchase_rate",
]
features[zero_cols] = features[zero_cols].fillna(0)

# For 'days' columns: dormant customers get worst-case value (999)
features["days_since_last_payment"]   = features["days_since_last_payment"].fillna(999)
features["avg_days_between_payments"] = features["avg_days_between_payments"].fillna(999)
features["recency_days"]    

0      313.0
1        NaN
2       27.0
3       37.0
4       18.0
       ...  
399      NaN
400      NaN
401    767.0
402    819.0
403      NaN
Name: recency_days, Length: 404, dtype: float64

 3.4  Derived risk metrics

In [12]:
# Payment/Purchase Ratio (PPR): fraction of invoiced amount that has been paid
features["payment_purchase_ratio"] = np.where(
    features["total_purchases"] > 0,
    (features["total_payments"] / features["total_purchases"]).clip(upper=1.50),
    0.0
)

# Net balance: total still owed (from the catalog field)
features["net_balance"] = features["TOTAL_POR_ABONAR"].clip(lower=0)

# Credit utilization: net balance vs. assigned credit limit
features["credit_utilization"] = np.where(
    features["LIMITE_DE_CREDITO"] > 0,
    (features["net_balance"] / features["LIMITE_DE_CREDITO"]).clip(upper=3.0),
    0.0
)

# Weighted Payment Frequency (WPF):
#   WPF = (payment_count * avg_payment) / (outstanding_balance + 1)
#   Higher value → more consistent, proportional repayments
features["weighted_payment_freq"] = (
    (features["payment_count"] * features["avg_payment"])
    / (features["net_balance"] + 1)
)

print(f"Master feature table: {features.shape[0]} customers × {features.shape[1]} columns")
features[[
    "CODIGO", "NOMBRE", "payment_purchase_ratio",
    "net_balance", "credit_utilization", "weighted_payment_freq", "recency_days"
]].head(6)


# List of columns to analyze
cols_to_summary = [
    "payment_purchase_ratio", 
    "net_balance", 
    "credit_utilization", 
    "weighted_payment_freq", 
    "recency_days"
]

# Create a summary DataFrame with the desired metrics
summary_df = features[cols_to_summary].agg(['min', 'max', 'mean']).transpose()

# Apply styling for a "visually attractive" look in the notebook
styled_summary = summary_df.style.format("{:.2f}") \
    .background_gradient(cmap='Blues', subset=['mean']) \
    .set_caption("Feature Distributions Summary") \
    .set_table_styles([
        {'selector': 'th', 'props': [('background-color', '#f4f4f4'), ('color', '#333'), ('font-weight', 'bold')]}
    ])

# Display the table
styled_summary

Master feature table: 404 customers × 28 columns


,min,max,mean
payment_purchase_ratio,0.00,1.50,0.53
net_balance,0.00,0.00,0.00
credit_utilization,0.00,0.00,0.00
weighted_payment_freq,-1000.00,56645637.42,2054997.53
recency_days,9.00,1137.00,313.36


In [13]:
features.to_excel("Clientes.xlsx",index=False)

3.5  NUEVAS MÉTRICAS ESTRATÉGICAS Y SCORING DEL CLIENTE

In [ ]:

# --- PARÁMETROS GLOBALES DEL NEGOCIO (¡Ajustables!) ---
MARGEN_OPERATIVO_EST = 0.20  # Asumimos 20% de utilidad tras deducciones base
TASA_INTERES_ANUAL   = 0.12  # Tasa de referencia del 12% para interés de capital retenido




# --- 1. EFICIENCIA FINANCIERA (UTILIDAD REAL) ---
total_ventas_empresa = features["total_purchases"].sum()
total_ventas_empresa = total_ventas_empresa if total_ventas_empresa > 0 else 1
features["concentracion_ventas_pct"] = (features["total_purchases"] / total_ventas_empresa) * 100
features["margen_operativo_bruto"] = features["total_purchases"] * MARGEN_OPERATIVO_EST
# Ticket Promedio (AOV) ya vive en la columna 'avg_invoice'
# --- 2. COMPORTAMIENTO DE PAGO (RIESGO Y FLUJO) ---
# DSO Implícito: Calculamos la proporción de su canasta mensual que debe, en días.
features["dso_individual_dias"] = np.where(
    features["monthly_purchase_rate"] > 0,
    (features["net_balance"] / features["monthly_purchase_rate"]) * 30.4,
    features["days_since_last_payment"]  # Proxy para cuentas congeladas
).clip(max=365)
# Costo Financiero de la Deuda: Penalización monetaria real anualizada por el retraso
features["costo_financiero_deuda"] = features["net_balance"] * (TASA_INTERES_ANUAL / 365) * features["dso_individual_dias"]
# --- 3. COSTO DE SERVICIO (COMPLEJIDAD OPERATIVA) ---
# Ratio de Devoluciones como erosión logística pura
features["devoluciones_pct"] = np.where(
    features["TOTAL_CREDITO"] > 0,
    (features["TOTAL_DEV_VENTA"] / features["TOTAL_CREDITO"]) * 100,
    0.0
).clip(max=100)


SyntaxError: '(' was never closed (616214744.py, line 10)

In [15]:
# =============================================================================
# 4. MATRIZ DE SCORING ESTRATÉGICO (0 a 100 PUNTOS)
# =============================================================================
# 4.1 Sub-Score Margen (Peso: 50%) -> Combina Utilidad Total con un Ticket Promedio Sano
score_margen_raw = features["margen_operativo_bruto"] + (features["avg_invoice"] * MARGEN_OPERATIVO_EST)
features["score_eficiencia_financiera"] = score_margen_raw.rank(pct=True) * 100
# 4.2 Sub-Score Pagos (Peso: 30%) -> Destruye el score si el DSO supera 120 días
features["score_comportamiento_pago"] = np.where(
    features["dso_individual_dias"] <= 120,
    (1 - (features["dso_individual_dias"] / 120)) * 100,
    0
)
# 4.3 Sub-Score Operativo (Peso: 20%) -> 30% de devoluciones tumba el score entero a 0
features["score_costo_servicio"] = np.where(
    features["devoluciones_pct"] < 30,
    100 - (features["devoluciones_pct"] * 3.33), 
    0
).clip(lower=0)
# --- SCORE GLOBAL (Sumativo Ponderado) ---
features["CLIENT_SCORE"] = (
    (features["score_eficiencia_financiera"] * 0.50) +
    (features["score_comportamiento_pago"] * 0.30) +
    (features["score_costo_servicio"] * 0.20)
)
# --- CATEGORIZACIÓN COMERCIAL ---
def assign_tier(score):
    if pd.isna(score): return "Sin Datos"
    if score >= 80: return "1 - VIP / Ancla"
    elif score >= 60: return "2 - Rentable Estándar"
    elif score >= 40: return "3 - En Fricción"
    else: return "4 - Tóxico / Fuga"
features["CLIENT_TIER"] = features["CLIENT_SCORE"].fillna(0).apply(assign_tier)
# Visualización rápida del Output
cols_reporte = ["NOMBRE", "CLIENT_TIER", "CLIENT_SCORE", "score_eficiencia_financiera", "dso_individual_dias", "costo_financiero_deuda", "devoluciones_pct"]
print("\n--- CLASIFICACIÓN DE CLIENTES (Scoring Estratégico) ---")
print(features[cols_reporte].sort_values(by="CLIENT_SCORE", ascending=False).head(15))

KeyError: 'devoluciones_pct'